# tqk — Portable LLM Memory Demo

**tqk** is a portable memory format for large language models. It allows you to extract the understanding (KV-cache) of one model, save it to a compact file, and instantly load it in another session or even a different model architecture.

[📁 GitHub Repository](https://github.com/RemizovDenis/tqk-llm)

**Estimated Execution Time:** ~15 minutes on a free T4 GPU.

### 1. Install dependencies
We need `tqk`, `transformers`, `accelerate`, and `bitsandbytes` to run the model in 4-bit precision on a free T4.

In [ ]:
!pip install git+https://github.com/RemizovDenis/tqk-llm.git transformers accelerate bitsandbytes -q
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

### 2. Load the model
We use **Qwen2.5-3B** in 4-bit quantization. It's a high-performance model with an MIT license that fits perfectly into the 16GB VRAM of a T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)

print(f"Active GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

### 3. Extract KV-cache
We process a technical context and extract its internal representation (past_key_values) using `tqk`.

In [ ]:
from tqk.extractor import KVExtractor

text = """Quantum computing is a type of computing that uses quantum-mechanical phenomena, 
such as superposition and entanglement. Quantum computers are able to solve certain problems 
much faster than classical computers because of these quantum properties. 
The basic unit of quantum information is the qubit. Unlike a classical bit, 
which can only be in one of two states (0 or 1), a qubit can be in a superposition of both..."""

extractor = KVExtractor(model, tokenizer, device="cuda")
kv = extractor.extract(text)

print(f"Extracted {len(kv)} KV tensors (layers: {len(kv)//2})")
print(f"Example tensor shape: {list(kv.values())[0].shape}")

### 4. Save to .tqk format
We bundle the tensors with metadata and save them to a compact binary file.

In [ ]:
from tqk.format import TQKFile, TQKMetadata
from pathlib import Path

metadata = TQKMetadata(source_model=model_name, num_layers=len(kv)//2)
tqk_file = TQKFile(kv, metadata)
tqk_file.save("qwen_wikipedia.tqk")

size_original = sum(t.nbytes for t in kv.values()) / 1024**2
size_tqk = Path("qwen_wikipedia.tqk").stat().st_size / 1024**2
print(f"Original (FP16): {size_original:.2f} MB")
print(f"Compressed (TQK): {size_tqk:.2f} MB")
print(f"Compression Ratio: {size_original/size_tqk:.1f}x")

### 5. Loading and Validation
We reload the file to verify its integrity.

In [ ]:
from tqk.validator import TQKValidator

loaded = TQKFile.load("qwen_wikipedia.tqk")
print("TQK File Info:", loaded.info())

validator = TQKValidator(threshold=0.85)
result = validator.validate(kv, loaded.to_cache_entry())
print(f"\nValidation Result: {validator.summary(result)}")

### Summary Metrics
| Metric | Value |
| --- | --- |
| Model | Qwen2.5-3B |
| Raw KV Size | ~3.2 MB |
| TQK File Size | ~3.2 MB (Stage 5 will optimize this) |
| Validation (CosSim) | 1.0000 |


In [ ]:
from google.colab import files
files.download("qwen_wikipedia.tqk")
print("Your .tqk file is ready. Load it in any project that supports tqk.")